# HCDS Depression Screener Project Python Data Analysis

---



### 1. Introduction and Data Merging
In this section, we load the two public-use SAS transport files (.xpt) from the 2021-2023 NHANES cycle. We merge the demographic variables (DEMO_L.xpt) with the mental health depression screener variables (DPQ_L.xpt) using the unique respondent sequence number (SEQN) as our primary merge key. An inner join is applied to keep only the participants who have entries in both datasets.

In [4]:
import pandas as pd
import numpy as np


#Load the SAS (.xpt) files using pandas
try:
    df_demo = pd.read_sas('DEMO_L.xpt')
    df_dpq = pd.read_sas('DPQ_L.xpt')
except FileNotFoundError as e:
    print(f" Error: File not found. Please verify paths. Details: {e}")
    raise e

print(f"Initial Demographic records: {df_demo.shape[0]}")
print(f"Initial Depression Screener records: {df_dpq.shape[0]}")

#Merge datasets on the unique participant identifier (SEQN)
df_merged = pd.merge(df_demo, df_dpq, on='SEQN', how='inner')
print(f"Merged Dataset Row Count (Intersection): {df_merged.shape[0]}\n")

Initial Demographic records: 11933
Initial Depression Screener records: 6337
Merged Dataset Row Count (Intersection): 6337



###2. Feature Selection & Subset Creation

To streamline our data framework, we extract only the attributes necessary for our clinical intake decision-making scenario. This includes specific sociodemographic predictors (e.g., gender, age, race, income-to-poverty ratio) and the nine individual items of the Patient Health Questionnaire (PHQ-9).

In [56]:
# Define our demographic features and target questionnaire items
demo_features = [
    'RIAGENDR', 'RIDAGEYR', 'RIDRETH3', 'DMDEDUC2',
    'DMDMARTZ', 'DMDBORN4', 'INDFMPIR', 'DMDHHSIZ'
]
dpq_items = [
    'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040',
    'DPQ050', 'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090'
]
all_needed_columns = ['SEQN'] + demo_features + dpq_items

# Create a clean subset dataframe
df_clean = df_merged[all_needed_columns].copy()
print(f"Subset created with {df_clean.shape[1]} clinical and demographic columns.")

Subset created with 18 clinical and demographic columns.


### 3. Handling Missing Values and Invalid Survey Codes
NHANES handles anomalies using specific placeholders: a value of 7 implies "Refused" and a value of 9 implies "Don't Know". These do not represent standard ordinal symptom frequency scores or demographic categories and would introduce severe bias. We recode them to standard NaN values.

Furthermore, we drop rows with incomplete PHQ-9 answers because a valid final score cannot be calculated if survey questions are skipped. Missing entries in our continuous economic indicator (INDFMPIR) are imputed using the median value to maintain sample stability.

In [57]:
# Convert all PHQ-9 items to numeric and replace values greater than 3 (like 7 and 9) with NaN
for col in dpq_items:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    df_clean.loc[df_clean[col] > 3, col] = np.nan

# Clean demographics by replacing 7 and 9 with NaN
for col in demo_features:
    if col != 'INDFMPIR':
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        df_clean[col] = df_clean[col].replace({7.0: np.nan, 9.0: np.nan})

# Drop rows only if they have missing values in the PHQ-9 questions
initial_rows = df_clean.shape[0]
df_clean = df_clean.dropna(subset=dpq_items)
dropped_rows = initial_rows - df_clean.shape[0]

print(f"Dropped {dropped_rows} records with missing/invalid PHQ-9 responses.")

# Handle missing values in income ratio using median imputation
if df_clean['INDFMPIR'].isnull().sum() > 0:
    median_val = df_clean['INDFMPIR'].median()
    df_clean['INDFMPIR'] = df_clean['INDFMPIR'].fillna(median_val)

Dropped 882 records with missing/invalid PHQ-9 responses.


## 4. Target Label Engineering (PHQ-9 Binarization)

In this step, we create our target label for the classification task. First, we sum the scores of the 9 individual depression questions (`DPQ010` to `DPQ090`) to find the total PHQ-9 score for each participant. This total score ranges from 0 to 27.

Following standard clinical guidelines, a total score of 10 or higher indicates a moderate-to-severe risk of depression. Therefore, we binarize this score to create our final `Target` variable: `1` for clinical depression risk ($\ge 10$) and `0` for low risk ($< 10$). This binary label will be the ground truth that our machine learning model tries to predict.

In [58]:
#Calculate the total continuous PHQ-9 diagnostic score
df_clean['PHQ9_Total'] = df_clean[dpq_items].sum(axis=1)

#Binarize the target label based on the standard clinical cutoff (Score >= 10)
df_clean['Target'] = (df_clean['PHQ9_Total'] >= 10).astype(int)

print(f"\nFinal Preprocessed Dataset Sample Size: {df_clean.shape[0]} participants.")
print(f"Positive Class (Depression Risk - Target=1): {df_clean['Target'].sum()} rows")
print(f"Negative Class (Low Risk - Target=0): {(df_clean['Target'] == 0).sum()} rows")
print(f"Class Imbalance Ratio (Base Rate): {(df_clean['Target'].mean() * 100):.2f}% positive")


Final Preprocessed Dataset Sample Size: 5455 participants.
Positive Class (Depression Risk - Target=1): 723 rows
Negative Class (Low Risk - Target=0): 4732 rows
Class Imbalance Ratio (Base Rate): 13.25% positive


## 5. Model Training and Evaluation (Random Forest)

In this stage, we split our preprocessed dataset into training (80%) and testing (20%) sets. Since our features contain categorical variables (such as gender and race), we apply an encoder to make them readable for the machine learning model.

We train a `RandomForestClassifier` using the `class_weight='balanced'` parameter. This setting is critical because our data has a significant class imbalance (only ~13% positive cases), and it forces the model to give equal weight to the minority group (distressed patients). To evaluate our model from a clinician's perspective, we skip standard Accuracy and instead calculate **Sensitivity (Recall)** to ensure we don't miss at-risk patients, **Specificity**, **Macro F1-Score**, and **AUC-ROC**.

In [60]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Define our features (X) and target label (y)
features = [
    'RIAGENDR', 'RIDAGEYR', 'RIDRETH3', 'DMDEDUC2',
    'DMDMARTZ', 'DMDBORN4', 'INDFMPIR', 'DMDHHSIZ'
]
X = df_clean[features].copy()
y = df_clean['Target'].copy()

# Identify categorical features that need encoding (except age, income ratio, and household size)
categorical_features = ['RIAGENDR', 'RIDRETH3', 'DMDEDUC2', 'DMDMARTZ', 'DMDBORN4']

# Apply OrdinalEncoder to transform categorical strings/codes into numerical formats
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[categorical_features] = encoder.fit_transform(X[categorical_features].astype(str))

# Split the dataset into 80% training and 20% testing sets
# stratify=y ensures both sets have the same 13.25% depression prevalence ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} patients")
print(f"Testing set size: {X_test.shape[0]} patients")

# Initialize Random Forest Classifier with class_weight='balanced' to handle class imbalance
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    max_depth=10
)

# Train the baseline model
rf_model.fit(X_train, y_train)

# Generate predictions and probability scores on the test set
y_pred = rf_model.predict(X_test)
y_probs = rf_model.predict_proba(X_test)[:, 1]

# Calculate Confusion Matrix elements to extract Specificity and Sensitivity
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
sensitivity = tp / (tp + fn)  # Equal to Recall
specificity = tn / (tn + fp)
auc_roc = roc_auc_score(y_test, y_probs)

# Display clinician-centric evaluation metrics
print(f"Sensitivity (Recall for At-Risk Class): {sensitivity:.4f}")
print(f"Specificity (Accuracy for Low-Risk Class): {specificity:.4f}")
print(f"AUC-ROC Score: {auc_roc:.4f}\n")

# Detailed classification report for precision, recall, and macro F1-score
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Risk (0)', 'At Risk (1)']))

Training set size: 4364 patients
Testing set size: 1091 patients
Sensitivity (Recall for At-Risk Class): 0.2828
Specificity (Accuracy for Low-Risk Class): 0.8636
AUC-ROC Score: 0.6603

Detailed Classification Report:
              precision    recall  f1-score   support

Low Risk (0)       0.89      0.86      0.88       946
 At Risk (1)       0.24      0.28      0.26       145

    accuracy                           0.79      1091
   macro avg       0.56      0.57      0.57      1091
weighted avg       0.80      0.79      0.79      1091



The baseline `RandomForestClassifier` trained successfully, but the resulting evaluation metrics highlight a critical challenge in predicting clinical depression using purely sociodemographic predictors.

### Key Observations & Technical Interpretation:
* **Severe False Negative Rate (FNR):** The model achieved a **Sensitivity (Recall)** of only `0.2828` on the minority class (`Target=1`). In a clinical screening scenario, this means the model misses approximately **71.72%** of patients who actually carry a moderate-to-severe risk of depression. From a patient safety perspective, a high FNR is highly problematic because vulnerable individuals would leave the clinic triage system undetected.
* **High Specificity vs. Low Precision:** While the model is highly capable of identifying non-depressed individuals (**Specificity** of `0.8636`), its **Precision** for the positive class is only `0.24`. This indicates that out of all patients flagged by the model as "At Risk," 76% are false alarms, which would trigger unnecessary secondary psychiatric evaluations and strain clinical workflows.
* **Discriminative Power (AUC-ROC):** The **AUC-ROC score** of `0.6603` confirms that the model’s ability to distinguish between high-risk and low-risk patients is only slightly better than a random guess ($0.50$).

### Justification for the Performance (Feature Limitation):
The bottleneck here is an issue of **insufficient signal in the feature space**. Predicting psychiatric conditions like major depressive disorder requires deep behavioral, physiological, or clinical biomarkers. Sociodemographic attributes (such as age, gender, race, and income-to-poverty ratio) provide macro-level statistical correlations across a population, but they lack the granular diagnostic resolution necessary to accurately capture individual psychological distress.


## 6. Fairness Auditing (False Negative Rate Parity)

In this stage, we evaluate whether our model exhibits systemic bias against specific demographic subgroups. For a clinician, missing a patient who actually needs help (**False Negative**) is the most dangerous error. Therefore, we use the **False Negative Rate (FNR)** as our primary fairness metric and audit it across the gender attribute (`RIAGENDR`).

We group the true test labels and the model's predictions by subgroup to calculate each group's FNR. Then, we measure the **FNR Disparity** (the absolute difference between groups). According to standard algorithmic fairness guidelines (like the 80% rule or parity thresholds), significant differences in FNR indicate that the model's predictive failures are unfairly concentrated on a specific protected group.

In [63]:

# Create a temporary dataframe combining true labels, predictions, and the raw sensitive attribute
# Note: Ensure X_test indices match df_clean to map back the original gender labels
test_indices = y_test.index
df_fairness = pd.DataFrame({
    'True_Label': y_test.values,
    'Prediction': y_pred,
    'Gender': df_clean.loc[test_indices, 'RIAGENDR'].values
})

# Map NHANES raw gender codes to readable strings for the clinician
# 1.0 is typically Male, 2.0 is Female in NHANES coding
df_fairness['Gender'] = df_fairness['Gender'].map({1.0: 'Male', 2.0: 'Female'})

# Dictionary to store calculated fairness metrics per subgroup
subgroup_metrics = {}

# Group by gender and compute False Negative Rate (FNR) for each
for gender, group in df_fairness.groupby('Gender'):
    # Confusion matrix elements for the current subgroup
    tp = ((group['True_Label'] == 1) & (group['Prediction'] == 1)).sum()
    fn = ((group['True_Label'] == 1) & (group['Prediction'] == 0)).sum()

    # FNR = FN / (TP + FN) -> Proportion of actual positive cases missed by the model
    fnr = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    subgroup_metrics[gender] = fnr

# Display the FNR per subgroup
for gender, fnr_val in subgroup_metrics.items():
    print(f"{gender} Subgroup - False Negative Rate (FNR): {fnr_val:.4f}")

# Calculate the Disparity (absolute difference between Male and Female FNR)
fnr_disparity = abs(subgroup_metrics['Male'] - subgroup_metrics['Female'])
print(f"\nAbsolute FNR Disparity: {fnr_disparity:.4f}")

Female Subgroup - False Negative Rate (FNR): 0.6224
Male Subgroup - False Negative Rate (FNR): 0.9149

Absolute FNR Disparity: 0.2924


Our test dataset contains exactly **1,091 patients**, out of which **145 patients** are ground-truth positive cases (`Target=1`). The breakdown of the model's False Negatives (FN) across subgroups perfectly accounts for the overall system behavior:
* **Female Subgroup:** Contains 98 actual positive cases. With an $FNR$ of `0.6224`, the model misses exactly **61 female patients** ($98 \times 0.6224$).
* **Male Subgroup:** Contains 47 actual positive cases. With an $FNR$ of `0.9149`, the model misses exactly **43 male patients** ($47 \times 0.9149$).


###Root Cause of the Disparity: Representation Bias
The massive **29.24% FNR Disparity** is a direct reflection of **Representation Bias** within the NHANES epidemiological cohort.

In public health data, clinical depression risk is statistically more prevalent and more explicitly self-reported by female participants. While using `class_weight='balanced'` forced the model to prioritize the minority target label (`Target=1`), the vast majority of the samples within that positive label belong to female patients. Consequently, the Random Forest model optimized its decision boundaries around the clinical manifestation patterns predominant in the female subpopulation, effectively leaving male depressive distress as a major systemic blind spot.

This finding demonstrates the exact purpose of Algorithmic Fairness auditing: uncovering how a technically sound model can inadvertently discriminate against a specific demographic group during clinical deployment.